In [1]:
!pip -q install sentence-transformers faiss-cpu pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.5 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import re
import json
import zipfile

import numpy as np
import pandas as pd
import faiss
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/step12")

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("GPU:", torch.cuda.is_available())
print("FAISS:", faiss.__version__)
print("OUTPUT:", OUT_DIR)

GPU: True
FAISS: 1.15.1
OUTPUT: /kaggle/working/step12


In [3]:
required_step11 = [
    "product_faiss.index",
    "product_embedding_metadata.parquet",
    "step11_config.json",
]

step11_files = {}

for filename in required_step11:
    matches = list(INPUT_DIR.rglob(filename))

    if matches:
        step11_files[filename] = matches[0]

missing = [
    filename
    for filename in required_step11
    if filename not in step11_files
]

if missing:

    zip_matches = list(
        INPUT_DIR.rglob("step11_final.zip")
    )

    assert zip_matches, (
        "Không tìm thấy file Step 11. "
        f"Đang thiếu: {missing}"
    )

    extract_dir = (
        Path("/kaggle/working")
        / "step11_extracted"
    )

    extract_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        zip_matches[0]
    ) as zf:

        zf.extractall(
            extract_dir
        )

    for filename in missing:

        matches = list(
            extract_dir.rglob(filename)
        )

        assert matches, (
            f"ZIP Step 11 thiếu {filename}"
        )

        step11_files[filename] = matches[0]

print("===== STEP 11 FILES =====")

for filename, path in step11_files.items():
    print(filename, "→", path)

===== STEP 11 FILES =====
product_faiss.index → /kaggle/input/datasets/trnquangtriu/12dataset/step11_final/product_faiss.index
product_embedding_metadata.parquet → /kaggle/input/datasets/trnquangtriu/12dataset/product_embedding_metadata.parquet
step11_config.json → /kaggle/input/datasets/trnquangtriu/12dataset/step11_config.json


In [4]:
required_step11 = [
    "product_faiss.index",
    "product_embedding_metadata.parquet",
    "step11_config.json",
]

step11_files = {}

for filename in required_step11:
    matches = list(INPUT_DIR.rglob(filename))

    if matches:
        step11_files[filename] = matches[0]

missing = [
    filename
    for filename in required_step11
    if filename not in step11_files
]

if missing:

    zip_matches = list(
        INPUT_DIR.rglob("step11_final.zip")
    )

    assert zip_matches, (
        "Không tìm thấy file Step 11. "
        f"Đang thiếu: {missing}"
    )

    extract_dir = (
        Path("/kaggle/working")
        / "step11_extracted"
    )

    extract_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        zip_matches[0]
    ) as zf:

        zf.extractall(
            extract_dir
        )

    for filename in missing:

        matches = list(
            extract_dir.rglob(filename)
        )

        assert matches, (
            f"ZIP Step 11 thiếu {filename}"
        )

        step11_files[filename] = matches[0]

print("===== STEP 11 FILES =====")

for filename, path in step11_files.items():
    print(filename, "→", path)

===== STEP 11 FILES =====
product_faiss.index → /kaggle/input/datasets/trnquangtriu/12dataset/step11_final/product_faiss.index
product_embedding_metadata.parquet → /kaggle/input/datasets/trnquangtriu/12dataset/product_embedding_metadata.parquet
step11_config.json → /kaggle/input/datasets/trnquangtriu/12dataset/step11_config.json


In [5]:
product_matches = list(
    INPUT_DIR.rglob(
        "unified_products.parquet"
    )
)

review_matches = list(
    INPUT_DIR.rglob(
        "unified_reviews.parquet"
    )
)

assert product_matches, (
    "Chưa Add Input dataset Step 10."
)

assert review_matches, (
    "Thiếu unified_reviews.parquet."
)

products = pd.read_parquet(
    product_matches[0]
)

reviews = pd.read_parquet(
    review_matches[0]
)

print("PRODUCTS:", products.shape)
print("REVIEWS:", reviews.shape)

assert len(products) == 164_926
assert len(reviews) == 767_179

assert products["product_id"].is_unique

print("\nPRODUCT COLUMNS:")
print(products.columns.tolist())

print("\nREVIEW COLUMNS:")
print(reviews.columns.tolist())

PRODUCTS: (164926, 60)
REVIEWS: (767179, 16)

PRODUCT COLUMNS:
['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'description', 'image_url', 'product_url', 'source', 'language', 'smart_product_name', 'smart_source', 'smart_currency', 'smart_price', 'smart_rating', 'smart_review_count', 'product_family', 'product_type', 'classification_method', 'matched_rule', 'has_valid_price', 'has_valid_currency', 'price_filter_eligible', 'quality_issue_count', 'quality_flags_json', 'product_family_v1', 'product_type_v1', 'classification_method_v1', 'matched_rule_v1', 'product_family_v2', 'product_type_v2', 'classification_method_v2', 'classification_reason_v2', 'etl_review_rows', 'etl_review_text_rows', 'etl_review_evidence_rows', 'has_review_evidence', 'product_url_available', 'product_type_candidate', 'product_family_candidate', 'classification_requires_validation', 'top5_evidence_ready', 'asin', 'meta_title', 'm

In [6]:
FAISS_PATH = step11_files[
    "product_faiss.index"
]

METADATA_PATH = step11_files[
    "product_embedding_metadata.parquet"
]

CONFIG_PATH = step11_files[
    "step11_config.json"
]

index = faiss.read_index(
    str(FAISS_PATH)
)

product_metadata = pd.read_parquet(
    METADATA_PATH
)

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    step11_config = json.load(f)

print("FAISS VECTORS:", index.ntotal)
print("FAISS DIMENSION:", index.d)
print("METADATA:", product_metadata.shape)

print("\nSTEP 11 CONFIG:")
print(
    json.dumps(
        step11_config,
        ensure_ascii=False,
        indent=2
    )
)

assert index.ntotal == len(
    product_metadata
)

assert index.d == 384

assert product_metadata[
    "faiss_row_id"
].tolist() == list(
    range(len(product_metadata))
)

assert product_metadata[
    "product_id"
].is_unique

print("\nPASS: FAISS mapping hợp lệ.")

FAISS VECTORS: 164926
FAISS DIMENSION: 384
METADATA: (164926, 17)

STEP 11 CONFIG:
{
  "step": "step11_product_embedding_faiss",
  "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "embedding_dimension": 384,
  "model_max_sequence_length": 128,
  "embedding_text_column": "embedding_text_v2",
  "normalization": true,
  "faiss_index_type": "IndexFlatIP",
  "similarity": "cosine",
  "total_products": 164926,
  "chunk_size": 4096,
  "batch_size": 64,
  "source_dataset": "step10_unified_products",
  "faiss_mapping_column": "faiss_row_id"
}

PASS: FAISS mapping hợp lệ.


In [7]:
step10_ids = set(
    products["product_id"]
)

step11_ids = set(
    product_metadata["product_id"]
)

print(
    "STEP 10 PRODUCTS:",
    len(step10_ids)
)

print(
    "STEP 11 PRODUCTS:",
    len(step11_ids)
)

print(
    "STEP 11 IDS MISSING IN STEP 10:",
    len(
        step11_ids - step10_ids
    )
)

print(
    "STEP 10 IDS MISSING IN STEP 11:",
    len(
        step10_ids - step11_ids
    )
)

assert step10_ids == step11_ids

print(
    "\nPASS: Step 10 và Step 11 "
    "có cùng tập product_id."
)

STEP 10 PRODUCTS: 164926
STEP 11 PRODUCTS: 164926
STEP 11 IDS MISSING IN STEP 10: 0
STEP 10 IDS MISSING IN STEP 11: 0

PASS: Step 10 và Step 11 có cùng tập product_id.


In [8]:
MODEL_NAME = step11_config[
    "embedding_model"
]

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

assert (
    model.get_sentence_embedding_dimension()
    == index.d
)

print("MODEL:", MODEL_NAME)
print("DEVICE:", DEVICE)
print("DIMENSION:", index.d)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MODEL: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
DEVICE: cuda
DIMENSION: 384


/tmp/ipykernel_58/216720882.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


In [9]:
def retrieve_candidates(
    query,
    top_k=100
):

    query_vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    scores, row_ids = index.search(
        query_vector,
        min(top_k, index.ntotal)
    )

    valid = row_ids[0] >= 0

    selected_ids = row_ids[0][valid]
    selected_scores = scores[0][valid]

    results = product_metadata.iloc[
        selected_ids
    ].copy()

    results["semantic_score"] = (
        selected_scores
    )

    return results.reset_index(
        drop=True
    )


query = (
    "Tôi cần thẻ nhớ microSD 32GB "
    "để lưu video và sử dụng cho điện thoại"
)

candidates = retrieve_candidates(
    query,
    top_k=100
)

print("QUERY:", query)
print("CANDIDATES:", len(candidates))

display(
    candidates[
        [
            "product_id",
            "product_name",
            "source",
            "price",
            "currency",
            "average_rating",
            "semantic_score"
        ]
    ].head(20)
)

QUERY: Tôi cần thẻ nhớ microSD 32GB để lưu video và sử dụng cho điện thoại
CANDIDATES: 100


,product_id,product_name,source,price,currency,average_rating,semantic_score
0,amazon_B074RNRM2B,SanDisk 400GB Ultra MicroSDXC UHS-I Memory Car...,amazon,62.20,USD,4.8,0.835000
1,amazon_B011BRUOMO,SanDisk Ultra 32GB microSDHC UHS-I Card with A...,amazon,8.99,USD,4.6,0.819043
2,amazon_B0BRZPCQNJ,SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memo...,amazon,16.99,USD,4.7,0.815231
3,amazon_B0BRZG5TK4,SanDisk 256GB Ultra microSDXC UHS-I Memory Car...,amazon,25.49,USD,4.7,0.814113
4,amazon_B073JWXGNT,SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card...,amazon,8.66,USD,4.7,0.813782
5,amazon_B0BFW1KJRS,Vansuny Micro SD Card 64GB microSDXC Memory Ca...,amazon,6.99,USD,4.4,0.793532
6,amazon_B09CGXP9RS,Micro Center 128GB Class 10 MicroSDXC Flash Me...,amazon,43.99,USD,4.7,0.787976
7,amazon_B097NJJPLY,SanDisk 8GB Mobile MicroSDHC Class 4 Flash Mem...,amazon,8.99,USD,4.5,0.783561
8,amazon_B07LG5WBTS,Micro Center 64GB Class 10 MicroSDXC Flash Mem...,amazon,7.99,USD,4.6,0.781600
9,amazon_B001L1H0SC,SanDisk 16 GB microSDHC Flash Memory Card SDSD...,amazon,7.00,USD,4.5,0.779855


In [10]:
import re

def extract_capacity_gb(query):
    """
    Trích xuất dung lượng GB/TB từ câu hỏi.
    Trả về GB hoặc None.
    """

    query = str(query).lower()

    pattern = (
        r"(?<!\d)"
        r"(\d+(?:[.,]\d+)?)"
        r"\s*"
        r"(tb|gb)"
        r"\b"
    )

    matches = re.findall(
        pattern,
        query,
        flags=re.IGNORECASE
    )

    if not matches:
        return None

    value, unit = matches[0]

    value = float(
        value.replace(",", ".")
    )

    if unit.lower() == "tb":
        return value * 1024

    return value


test_queries = [
    "Tôi cần thẻ nhớ microSD 32GB",
    "Tìm RAM 16 GB",
    "Ổ cứng SSD 1TB",
    "Tai nghe bluetooth giá rẻ",
]

for q in test_queries:
    print(
        q,
        "→",
        extract_capacity_gb(q)
    )

Tôi cần thẻ nhớ microSD 32GB → 32.0
Tìm RAM 16 GB → 16.0
Ổ cứng SSD 1TB → 1024.0
Tai nghe bluetooth giá rẻ → None


In [11]:
def extract_product_capacities_gb(
    product_name
):
    """
    Trả về danh sách dung lượng GB/TB
    tìm thấy trong tên sản phẩm.
    """

    if pd.isna(product_name):
        return []

    text = str(product_name).lower()

    pattern = (
        r"(?<!\d)"
        r"(\d+(?:[.,]\d+)?)"
        r"\s*"
        r"(tb|gb)"
        r"\b"
    )

    matches = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    capacities = []

    for value, unit in matches:

        number = float(
            value.replace(",", ".")
        )

        if unit.lower() == "tb":
            number *= 1024

        capacities.append(number)

    return capacities


examples = [
    "SanDisk 32GB 2-Pack Ultra MicroSD",
    "SanDisk 64GB Ultra MicroSD",
    "Samsung SSD 1TB",
    "Apple AirPods 2nd Generation",
]

for name in examples:
    print(
        name,
        "→",
        extract_product_capacities_gb(name)
    )

SanDisk 32GB 2-Pack Ultra MicroSD → [32.0]
SanDisk 64GB Ultra MicroSD → [64.0]
Samsung SSD 1TB → [1024.0]
Apple AirPods 2nd Generation → []


In [12]:
def filter_by_capacity(
    candidates,
    requested_capacity_gb
):

    if requested_capacity_gb is None:
        return candidates.copy()

    result = candidates.copy()

    result["detected_capacities_gb"] = (
        result["product_name"].apply(
            extract_product_capacities_gb
        )
    )

    mask = result[
        "detected_capacities_gb"
    ].apply(
        lambda capacities: any(
            abs(
                capacity
                - requested_capacity_gb
            ) < 0.001
            for capacity in capacities
        )
    )

    return result.loc[
        mask
    ].copy().reset_index(
        drop=True
    )


requested_capacity = extract_capacity_gb(
    query
)

filtered_candidates = filter_by_capacity(
    candidates,
    requested_capacity
)

print(
    "REQUESTED CAPACITY:",
    requested_capacity,
    "GB"
)

print(
    "BEFORE FILTER:",
    len(candidates)
)

print(
    "AFTER FILTER:",
    len(filtered_candidates)
)

display(
    filtered_candidates[
        [
            "product_name",
            "price",
            "currency",
            "average_rating",
            "semantic_score",
            "detected_capacities_gb"
        ]
    ].head(10)
)

REQUESTED CAPACITY: 32.0 GB
BEFORE FILTER: 100
AFTER FILTER: 28


,product_name,price,currency,average_rating,semantic_score,detected_capacities_gb
0,SanDisk Ultra 32GB microSDHC UHS-I Card with A...,8.99,USD,4.6,0.819043,[32.0]
1,SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card...,8.66,USD,4.7,0.813782,[32.0]
2,"Hugolog 32GB Micro SD Card, MicroSDXC UHS-I Me...",7.99,USD,4.5,0.779443,[32.0]
3,SanDisk Flash 32 GB microSDHC Flash Memory Car...,7.99,USD,4.6,0.764971,[32.0]
4,SanDisk Ultra SDSQUNS-032G-GN3MN 32GB 80MB/s U...,7.15,USD,4.7,0.762015,[32.0]
5,ADATA Premier 32GB microSDHC/SDXC UHS-I Class ...,8.98,USD,4.4,0.760772,[32.0]
6,Verbatim 32GB Premium microSDHC Memory Card wi...,6.97,USD,4.6,0.759611,[32.0]
7,"ADROITLARK 32GB Memory Card + SD Adapter, Micr...",6.99,USD,4.5,0.756276,"[32.0, 32.0]"
8,BRAVEEAGLE FAT32 32GB Micro SD Card Class 10 m...,6.99,USD,4.6,0.755689,"[32.0, 32.0]"
9,"LaView 32GB Micro SD Card 2 Pack, Micro SDXC U...",14.49,USD,4.6,0.753182,[32.0]


In [13]:
assert requested_capacity == 32.0

assert len(filtered_candidates) > 0, (
    "Không tìm thấy sản phẩm 32GB "
    "trong Top 100 ứng viên."
)

assert filtered_candidates[
    "detected_capacities_gb"
].apply(
    lambda values: 32.0 in values
).all()

print(
    "PASS: Tất cả ứng viên "
    "đều có dung lượng 32GB."
)

PASS: Tất cả ứng viên đều có dung lượng 32GB.


In [14]:
import re

def extract_budget(query):
    text = str(query).lower()

    currency_pattern = (
        r"(usd|vnd|twd|đồng|vnđ|₫|\$)"
    )

    currency_match = re.search(
        currency_pattern,
        text,
        flags=re.IGNORECASE
    )

    if not currency_match:
        return {
            "max_price": None,
            "currency": None
        }

    currency_raw = currency_match.group(1).lower()

    currency_map = {
        "usd": "USD",
        "$": "USD",
        "vnd": "VND",
        "vnđ": "VND",
        "đồng": "VND",
        "₫": "VND",
        "twd": "TWD",
    }

    currency = currency_map[currency_raw]

    pattern = (
        r"(?:dưới|không quá|tối đa|under|max)"
        r"\s*"
        r"(\d+(?:[.,]\d+)?)"
        r"\s*"
        r"(triệu|nghìn|ngàn|k)?"
        r"\s*"
        r"(?:usd|vnd|twd|đồng|vnđ|₫|\$)"
    )

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if not match:
        return {
            "max_price": None,
            "currency": currency
        }

    amount = float(
        match.group(1).replace(",", ".")
    )

    multiplier = match.group(2)

    if multiplier in ["nghìn", "ngàn", "k"]:
        amount *= 1000

    elif multiplier == "triệu":
        amount *= 1_000_000

    return {
        "max_price": amount,
        "currency": currency
    }


tests = [
    "Thẻ nhớ 32GB dưới 15 USD",
    "Tai nghe dưới 50 USD",
    "Điện thoại dưới 5 triệu VND",
    "Laptop tối đa 1000 USD",
    "Tôi cần thẻ nhớ 32GB",
]

for test in tests:
    print(test, "→", extract_budget(test))

Thẻ nhớ 32GB dưới 15 USD → {'max_price': 15.0, 'currency': 'USD'}
Tai nghe dưới 50 USD → {'max_price': 50.0, 'currency': 'USD'}
Điện thoại dưới 5 triệu VND → {'max_price': 5000000.0, 'currency': 'VND'}
Laptop tối đa 1000 USD → {'max_price': 1000.0, 'currency': 'USD'}
Tôi cần thẻ nhớ 32GB → {'max_price': None, 'currency': None}


In [15]:
def filter_by_budget(
    candidates,
    max_price=None,
    currency=None
):
    result = candidates.copy()

    if currency is not None:
        result = result[
            result["currency"]
            .astype(str)
            .str.upper()
            .eq(currency)
        ].copy()

    if max_price is not None:
        result["price_numeric"] = (
            pd.to_numeric(
                result["price"],
                errors="coerce"
            )
        )

        result = result[
            result["price_numeric"].notna()
            &
            result["price_numeric"].ge(0)
            &
            result["price_numeric"].le(max_price)
        ].copy()

    return result.reset_index(drop=True)


test_query = (
    "Tôi cần thẻ nhớ microSD 32GB "
    "dưới 15 USD để lưu video "
    "và sử dụng cho điện thoại"
)

budget = extract_budget(test_query)

print("BUDGET:", budget)

capacity = extract_capacity_gb(
    test_query
)

semantic_candidates = retrieve_candidates(
    test_query,
    top_k=100
)

capacity_candidates = filter_by_capacity(
    semantic_candidates,
    capacity
)

budget_candidates = filter_by_budget(
    capacity_candidates,
    max_price=budget["max_price"],
    currency=budget["currency"]
)

print(
    "SEMANTIC CANDIDATES:",
    len(semantic_candidates)
)

print(
    "AFTER CAPACITY:",
    len(capacity_candidates)
)

print(
    "AFTER BUDGET:",
    len(budget_candidates)
)

display(
    budget_candidates[
        [
            "product_name",
            "price",
            "currency",
            "average_rating",
            "semantic_score"
        ]
    ].head(10)
)

BUDGET: {'max_price': 15.0, 'currency': 'USD'}
SEMANTIC CANDIDATES: 100
AFTER CAPACITY: 30
AFTER BUDGET: 24


,product_name,price,currency,average_rating,semantic_score
0,SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card...,8.66,USD,4.7,0.787277
1,SanDisk Ultra 32GB microSDHC UHS-I Card with A...,8.99,USD,4.6,0.779868
2,BRAVEEAGLE FAT32 32GB Micro SD Card Class 10 m...,6.99,USD,4.6,0.749779
3,"Hugolog 32GB Micro SD Card, MicroSDXC UHS-I Me...",7.99,USD,4.5,0.746471
4,ADATA Premier 32GB microSDHC/SDXC UHS-I Class ...,8.98,USD,4.4,0.733113
5,SanDisk Ultra SDSQUNS-032G-GN3MN 32GB 80MB/s U...,7.15,USD,4.7,0.725661
6,32GB microSD Memory Card Micro SD Compatible w...,14.99,USD,4.6,0.724295
7,AEGO Micro SD Card 32GB Mini sd Card UHS-1 Cla...,6.99,USD,4.5,0.721608
8,Verbatim 32GB Premium microSDHC Memory Card wi...,6.97,USD,4.6,0.718353
9,SanDisk Flash 32 GB microSDHC Flash Memory Car...,7.99,USD,4.6,0.717961


In [16]:
def rank_products(
    candidates,
    top_k=5
):
    result = candidates.copy()

    if result.empty:
        return result

    result["semantic_component"] = (
        pd.to_numeric(
            result["semantic_score"],
            errors="coerce"
        )
        .fillna(0)
        .clip(0, 1)
    )

    rating = pd.to_numeric(
        result["average_rating"],
        errors="coerce"
    )

    result["rating_component"] = (
        rating
        .div(5)
        .clip(0, 1)
        .fillna(0)
    )

    review_count = pd.to_numeric(
        result["review_count"],
        errors="coerce"
    ).fillna(0).clip(lower=0)

    result["review_component"] = (
        np.log1p(review_count)
        /
        np.log1p(
            max(
                float(review_count.max()),
                1.0
            )
        )
    )

    result["final_score"] = (
        0.75 * result["semantic_component"]
        +
        0.20 * result["rating_component"]
        +
        0.05 * result["review_component"]
    )

    result = result.sort_values(
        by=[
            "final_score",
            "semantic_score",
            "product_id"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )

    return result.head(
        top_k
    ).reset_index(drop=True)


top5 = rank_products(
    budget_candidates,
    top_k=5
)

print("TOP 5 PRODUCTS:", len(top5))

display(
    top5[
        [
            "product_id",
            "product_name",
            "price",
            "currency",
            "average_rating",
            "review_count",
            "semantic_score",
            "final_score"
        ]
    ]
)

TOP 5 PRODUCTS: 5


,product_id,product_name,price,currency,average_rating,review_count,semantic_score,final_score
0,amazon_B073JWXGNT,SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card...,8.66,USD,4.7,233718,0.787277,0.828458
1,amazon_B011BRUOMO,SanDisk Ultra 32GB microSDHC UHS-I Card with A...,8.99,USD,4.6,115781,0.779868,0.816060
2,amazon_B0BZ4MFNY1,BRAVEEAGLE FAT32 32GB Micro SD Card Class 10 m...,6.99,USD,4.6,4586,0.749779,0.780435
3,amazon_B074B28H3S,SanDisk Ultra SDSQUNS-032G-GN3MN 32GB 80MB/s U...,7.15,USD,4.7,19323,0.725661,0.772164
4,amazon_B016A5NH6C,ADATA Premier 32GB microSDHC/SDXC UHS-I Class ...,8.98,USD,4.4,16578,0.733113,0.765133


In [17]:
assert len(top5) <= 5

assert (
    top5["currency"] == "USD"
).all()

assert (
    pd.to_numeric(
        top5["price"],
        errors="coerce"
    ) <= 15
).all()

assert top5[
    "product_name"
].apply(
    lambda name: 32.0 in
    extract_product_capacities_gb(name)
).all()

print(
    "PASS: Top 5 tuân thủ "
    "dung lượng 32GB và ngân sách 15 USD."
)

PASS: Top 5 tuân thủ dung lượng 32GB và ngân sách 15 USD.


In [18]:
import re

def extract_budget(query):
    text = str(query).lower().strip()

    pattern = (
        r"(?:dưới|không quá|tối đa|under|max)"
        r"\s*"
        r"(\d+(?:[.,]\d+)?)"
        r"\s*"
        r"(triệu|nghìn|ngàn|k)?"
        r"\s*"
        r"(usd|vnd|twd|vnđ|đồng|₫|\$)?"
    )

    match = re.search(pattern, text)

    if not match:
        return {
            "max_price": None,
            "currency": None,
            "needs_currency_clarification": False
        }

    amount = float(
        match.group(1).replace(",", ".")
    )

    multiplier = match.group(2)
    currency_raw = match.group(3)

    if multiplier in ["nghìn", "ngàn", "k"]:
        amount *= 1000

    elif multiplier == "triệu":
        amount *= 1_000_000

    currency_map = {
        "usd": "USD",
        "$": "USD",
        "vnd": "VND",
        "vnđ": "VND",
        "đồng": "VND",
        "₫": "VND",
        "twd": "TWD",
    }

    currency = (
        currency_map.get(currency_raw)
        if currency_raw
        else None
    )

    return {
        "max_price": amount,
        "currency": currency,
        "needs_currency_clarification": (
            currency is None
        )
    }


tests = [
    "Thẻ nhớ 32GB dưới 15 USD",
    "Tai nghe dưới 50 USD",
    "Điện thoại dưới 5 triệu VND",
    "Điện thoại dưới 5 triệu",
    "Thẻ nhớ dưới 500 nghìn đồng",
    "Laptop tối đa 1000 USD",
    "Tôi cần thẻ nhớ 32GB",
]

for q in tests:
    print(q, "→", extract_budget(q))

Thẻ nhớ 32GB dưới 15 USD → {'max_price': 15.0, 'currency': 'USD', 'needs_currency_clarification': False}
Tai nghe dưới 50 USD → {'max_price': 50.0, 'currency': 'USD', 'needs_currency_clarification': False}
Điện thoại dưới 5 triệu VND → {'max_price': 5000000.0, 'currency': 'VND', 'needs_currency_clarification': False}
Điện thoại dưới 5 triệu → {'max_price': 5000000.0, 'currency': None, 'needs_currency_clarification': True}
Thẻ nhớ dưới 500 nghìn đồng → {'max_price': 500000.0, 'currency': 'VND', 'needs_currency_clarification': False}
Laptop tối đa 1000 USD → {'max_price': 1000.0, 'currency': 'USD', 'needs_currency_clarification': False}
Tôi cần thẻ nhớ 32GB → {'max_price': None, 'currency': None, 'needs_currency_clarification': False}


In [19]:
def recommend_products(
    user_query,
    top_k=5,
    candidate_k=200
):
    # 1. Phân tích yêu cầu
    capacity = extract_capacity_gb(
        user_query
    )

    budget = extract_budget(
        user_query
    )

    # Không tự suy đoán đơn vị tiền tệ
    if budget[
        "needs_currency_clarification"
    ]:
        return {
            "query": user_query,
            "status": "needs_clarification",
            "message": (
                "Bạn muốn ngân sách "
                f"{budget['max_price']:,.0f} "
                "theo đơn vị tiền tệ nào "
                "(VND, USD hoặc TWD)?"
            ),
            "products": pd.DataFrame()
        }

    # 2. FAISS lấy ứng viên
    candidates = retrieve_candidates(
        user_query,
        top_k=candidate_k
    )

    # 3. Lọc dung lượng
    candidates = filter_by_capacity(
        candidates,
        capacity
    )

    # 4. Lọc ngân sách
    candidates = filter_by_budget(
        candidates,
        max_price=budget["max_price"],
        currency=budget["currency"]
    )

    # 5. Không đủ ứng viên
    if candidates.empty:
        return {
            "query": user_query,
            "status": "no_match",
            "message": (
                "Chưa tìm thấy sản phẩm "
                "đáp ứng các điều kiện "
                "trong tập ứng viên đã truy xuất."
            ),
            "products": pd.DataFrame()
        }

    # 6. Xếp hạng
    ranked = rank_products(
        candidates,
        top_k=top_k
    )

    status = (
        "success"
        if len(ranked) == top_k
        else "partial"
    )

    return {
        "query": user_query,
        "status": status,
        "message": (
            f"Tìm thấy {len(ranked)} "
            "sản phẩm phù hợp với "
            "các bộ lọc hiện tại."
        ),
        "products": ranked
    }

In [20]:
test_queries = [
    (
        "Tôi cần thẻ nhớ microSD 32GB "
        "dưới 15 USD để lưu video"
    ),

    (
        "Tôi cần thẻ nhớ microSD 32GB "
        "để dùng cho điện thoại"
    ),

    (
        "Tôi cần thẻ nhớ microSD 32GB "
        "dưới 500 nghìn đồng"
    ),

    (
        "Tôi cần thẻ nhớ microSD 32GB "
        "dưới 500 nghìn"
    ),
]

for i, user_query in enumerate(
    test_queries,
    start=1
):

    print("\n" + "=" * 70)
    print("TEST:", i)
    print("QUERY:", user_query)

    result = recommend_products(
        user_query,
        top_k=5,
        candidate_k=200
    )

    print("STATUS:", result["status"])
    print("MESSAGE:", result["message"])

    if not result["products"].empty:

        display(
            result["products"][
                [
                    "product_id",
                    "product_name",
                    "price",
                    "currency",
                    "average_rating",
                    "final_score"
                ]
            ]
        )


TEST: 1
QUERY: Tôi cần thẻ nhớ microSD 32GB dưới 15 USD để lưu video
STATUS: success
MESSAGE: Tìm thấy 5 sản phẩm phù hợp với các bộ lọc hiện tại.


,product_id,product_name,price,currency,average_rating,final_score
0,amazon_B073JWXGNT,SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card...,8.66,USD,4.7,0.798381
1,amazon_B011BRUOMO,SanDisk Ultra 32GB microSDHC UHS-I Card with A...,8.99,USD,4.6,0.786037
2,amazon_B0C2VMSBGX,"Hugolog 32GB Micro SD Card, MicroSDXC UHS-I Me...",7.99,USD,4.5,0.772212
3,amazon_B016A5NH6C,ADATA Premier 32GB microSDHC/SDXC UHS-I Class ...,8.98,USD,4.4,0.767298
4,amazon_B0BVVJM2X8,"LaView 32GB Micro SD Card 2 Pack, Micro SDXC U...",14.49,USD,4.6,0.764532



TEST: 2
QUERY: Tôi cần thẻ nhớ microSD 32GB để dùng cho điện thoại
STATUS: success
MESSAGE: Tìm thấy 5 sản phẩm phù hợp với các bộ lọc hiện tại.


,product_id,product_name,price,currency,average_rating,final_score
0,amazon_B073JWXGNT,SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card...,8.66,USD,4.7,0.838469
1,amazon_B0BZ4MFNY1,BRAVEEAGLE FAT32 32GB Micro SD Card Class 10 m...,6.99,USD,4.6,0.817052
2,amazon_B011BRUOMO,SanDisk Ultra 32GB microSDHC UHS-I Card with A...,8.99,USD,4.6,0.811640
3,amazon_B07T157TSH,SanDisk 32GB SDHC Micro Ultra Memory Card Work...,11.96,USD,4.6,0.807179
4,amazon_B07SYM1PP1,SanDisk 32GB SDHC Micro Ultra Memory Card Work...,9.90,USD,4.6,0.805496



TEST: 3
QUERY: Tôi cần thẻ nhớ microSD 32GB dưới 500 nghìn đồng
STATUS: no_match
MESSAGE: Chưa tìm thấy sản phẩm đáp ứng các điều kiện trong tập ứng viên đã truy xuất.

TEST: 4
QUERY: Tôi cần thẻ nhớ microSD 32GB dưới 500 nghìn
STATUS: needs_clarification
MESSAGE: Bạn muốn ngân sách 500,000 theo đơn vị tiền tệ nào (VND, USD hoặc TWD)?


In [23]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

cf_filenames = [
    "bpr_model.pkl",
    "cf_mappings.pkl",
    "cf_config.json",
    "hybrid_config.json",
]

print("===== COLLABORATIVE FILTERING FILES =====")

cf_files = {}

for filename in cf_filenames:

    matches = list(
        INPUT_DIR.rglob(filename)
    )

    if matches:
        cf_files[filename] = matches[0]

        print(
            "FOUND:",
            filename,
            "→",
            matches[0]
        )

    else:
        print(
            "NOT FOUND:",
            filename
        )

===== COLLABORATIVE FILTERING FILES =====
FOUND: bpr_model.pkl → /kaggle/input/datasets/trnquangtriu/12bdataset/bpr_model.pkl
FOUND: cf_mappings.pkl → /kaggle/input/datasets/trnquangtriu/12bdataset/cf_mappings.pkl
FOUND: cf_config.json → /kaggle/input/datasets/trnquangtriu/12bdataset/cf_config.json
FOUND: hybrid_config.json → /kaggle/input/datasets/trnquangtriu/12bdataset/hybrid_config.json


In [24]:
import json

for filename in [
    "cf_config.json",
    "hybrid_config.json"
]:
    path = cf_files[filename]

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        config = json.load(f)

    print("\n" + "=" * 60)
    print(filename)
    print("=" * 60)

    print(
        json.dumps(
            config,
            ensure_ascii=False,
            indent=2
        )
    )


cf_config.json
{
  "model_type": "BayesianPersonalizedRanking",
  "selected_model": "BPR_F64_R05",
  "selection_metric": "Validation NDCG@10",
  "factors": 64,
  "regularization": 0.05,
  "iterations": 50,
  "learning_rate": 0.05,
  "train_users": 26903,
  "train_products": 15546,
  "train_interactions": 205621,
  "test_users": 26903,
  "test_known_targets": 26585,
  "test_cold_targets": 318
}

hybrid_config.json
{
  "selection_dataset": "validation",
  "evaluation_dataset": "test",
  "candidate_strategy": "train_only_popularity_top_200",
  "candidate_count": 200,
  "models": {
    "Popularity": {
      "semantic": 0.0,
      "bpr": 0.0,
      "popularity": 1.0
    },
    "PopBPR_10": {
      "semantic": 0.0,
      "bpr": 0.1,
      "popularity": 0.9
    },
    "PopSemBPR_C": {
      "semantic": 0.1,
      "bpr": 0.1,
      "popularity": 0.8
    }
  },
  "evaluation_ks": [
    5,
    10,
    20
  ],
  "selection_metrics": [
    "Recall@10",
    "NDCG@10"
  ]
}


In [26]:
!pip -q install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 96.4 MB/s eta 0:00:00:00:0100:01


In [28]:
import pickle
import implicit

# Chỉ unpickle file do chính bạn tạo ở các Step trước.

with open(
    cf_files["cf_mappings.pkl"],
    "rb"
) as f:
    cf_mapping = pickle.load(f)

with open(
    cf_files["bpr_model.pkl"],
    "rb"
) as f:
    bpr_model = pickle.load(f)

print("===== CF MAPPING =====")
print("TYPE:", type(cf_mapping))

if isinstance(cf_mapping, dict):
    print("KEYS:", list(cf_mapping.keys()))

    for key, value in cf_mapping.items():
        print(
            key,
            "| type:", type(value),
            "| length:",
            len(value)
            if hasattr(value, "__len__")
            else "N/A"
        )

print("\n===== BPR MODEL =====")
print("TYPE:", type(bpr_model))

for attr in [
    "user_factors",
    "item_factors"
]:
    value = getattr(bpr_model, attr, None)

    if value is not None:
        print(attr, "| shape:", value.shape)

print("\nPASS: Đọc mô hình BPR thành công.")

===== CF MAPPING =====
TYPE: <class 'dict'>
KEYS: ['user_to_idx', 'item_to_idx', 'idx_to_user', 'idx_to_item']
user_to_idx | type: <class 'dict'> | length: 26903
item_to_idx | type: <class 'dict'> | length: 15546
idx_to_user | type: <class 'numpy.ndarray'> | length: 26903
idx_to_item | type: <class 'numpy.ndarray'> | length: 15546

===== BPR MODEL =====
TYPE: <class 'implicit.cpu.bpr.BayesianPersonalizedRanking'>
user_factors | shape: (26903, 65)
item_factors | shape: (15546, 65)

PASS: Đọc mô hình BPR thành công.


In [29]:
import json
import pandas as pd

test_records = []

for test_id, user_query in enumerate(
    test_queries,
    start=1
):

    result = recommend_products(
        user_query,
        top_k=5,
        candidate_k=200
    )

    test_records.append({
        "test_id": test_id,
        "query": user_query,
        "status": result["status"],
        "message": result["message"],
        "returned_products": len(
            result["products"]
        ),
        "product_ids": (
            result["products"]["product_id"]
            .tolist()
            if not result["products"].empty
            else []
        )
    })

test_results = pd.DataFrame(
    test_records
)

TEST_PATH = (
    OUT_DIR / "step12_test_results.json"
)

with open(
    TEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        test_records,
        f,
        ensure_ascii=False,
        indent=2
    )

display(test_results)

print("SAVED:", TEST_PATH)

,test_id,query,status,message,returned_products,product_ids
0,1,Tôi cần thẻ nhớ microSD 32GB dưới 15 USD để lư...,success,Tìm thấy 5 sản phẩm phù hợp với các bộ lọc hiệ...,5,"[amazon_B073JWXGNT, amazon_B011BRUOMO, amazon_..."
1,2,Tôi cần thẻ nhớ microSD 32GB để dùng cho điện ...,success,Tìm thấy 5 sản phẩm phù hợp với các bộ lọc hiệ...,5,"[amazon_B073JWXGNT, amazon_B0BZ4MFNY1, amazon_..."
2,3,Tôi cần thẻ nhớ microSD 32GB dưới 500 nghìn đồng,no_match,Chưa tìm thấy sản phẩm đáp ứng các điều kiện t...,0,[]
3,4,Tôi cần thẻ nhớ microSD 32GB dưới 500 nghìn,needs_clarification,"Bạn muốn ngân sách 500,000 theo đơn vị tiền tệ...",0,[]


SAVED: /kaggle/working/step12/step12_test_results.json


In [30]:
print("===== USER ID SAMPLE =====")

user_ids = list(
    cf_mapping["user_to_idx"].keys()
)

print(user_ids[:10])

print("\n===== PRODUCT ID SAMPLE =====")

item_ids = list(
    cf_mapping["item_to_idx"].keys()
)

print(item_ids[:20])

print("\n===== STEP 10 PRODUCT ID SAMPLE =====")

print(
    products["product_id"]
    .head(20)
    .tolist()
)

===== USER ID SAMPLE =====
['amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA', 'amazon_AE22MRZKBD5MWJLIMGYLLJ67ZWVQ', 'amazon_AE22PJ54OVIRX3I6KSLMPRHPHA4A', 'amazon_AE22U2DM2E4EOEDEEZ46BDVIYPOQ', 'amazon_AE22Z3RLVIRU6RT5PNRK5CFFNEFQ', 'amazon_AE232BKM6GQ5F2J6VSBCVGVG5YNQ', 'amazon_AE232P2BO4KMNTMGZIEKMQCZJAGA', 'amazon_AE2354O5OHFEFYH6IL7KWZOBG3EA', 'amazon_AE23AQBETJKYRHKGBYS5D6DQJ3BQ', 'amazon_AE23CIZ4OTQEFKSROZ3RHWCKBUIQ']

===== PRODUCT ID SAMPLE =====
['amazon_B0007LCLPE', 'amazon_B08KHM6QJB', 'amazon_B00U3FPN4U', 'amazon_B00BIDK1WU', 'amazon_B0787HCKR2', 'amazon_B00DK0VBTQ', 'amazon_B00UR321B6', 'amazon_B07VGGTPNQ', 'amazon_B01LX5UXYU', 'amazon_B00PXKK9BM', 'amazon_B0C5V1W3XV', 'amazon_B00005T3GH', 'amazon_B00FOHDK7O', 'amazon_B00BONB8FS', 'amazon_B016XZU3IY', 'amazon_B08XNCHTCY', 'amazon_B0C4LF1T4X', 'amazon_B0B3JQ1HMZ', 'amazon_B08L9M826M', 'amazon_B071XY12YY']

===== STEP 10 PRODUCT ID SAMPLE =====
['amazon_B07PXGQC1Q', 'amazon_B07KTYJ769', 'amazon_B0BGNG1294', 'amazon_B08CLNX58K', 'amazo

In [31]:
bpr_item_ids = set(
    cf_mapping["item_to_idx"].keys()
)

step10_item_ids = set(
    products["product_id"]
)

matched_ids = (
    bpr_item_ids
    &
    step10_item_ids
)

print(
    "BPR ITEMS:",
    len(bpr_item_ids)
)

print(
    "STEP 10 PRODUCTS:",
    len(step10_item_ids)
)

print(
    "MATCHED PRODUCTS:",
    len(matched_ids)
)

print(
    "BPR COVERAGE IN STEP 10:",
    round(
        len(matched_ids)
        /
        len(step10_item_ids)
        * 100,
        2
    ),
    "%"
)

print(
    "BPR ITEMS FOUND IN STEP 10:",
    round(
        len(matched_ids)
        /
        len(bpr_item_ids)
        * 100,
        2
    ),
    "%"
)

print("\nMATCHED EXAMPLES:")

print(
    list(matched_ids)[:10]
)

BPR ITEMS: 15546
STEP 10 PRODUCTS: 164926
MATCHED PRODUCTS: 15546
BPR COVERAGE IN STEP 10: 9.43 %
BPR ITEMS FOUND IN STEP 10: 100.0 %

MATCHED EXAMPLES:
['amazon_B00R1323WM', 'amazon_B082VQQKMZ', 'amazon_B0BRS4KZ93', 'amazon_B0BTBTFRNK', 'amazon_B088Y2T82G', 'amazon_B099WGJR62', 'amazon_B00IAKLM54', 'amazon_B09674F2X6', 'amazon_B07P6DW1RB', 'amazon_B07CPKBY82']


In [32]:
top5_ids = set(
    top5["product_id"]
)

matched_top5 = (
    top5_ids
    &
    bpr_item_ids
)

print(
    "TOP 5 PRODUCTS:",
    len(top5_ids)
)

print(
    "TOP 5 IN BPR:",
    len(matched_top5)
)

print(
    "TOP 5 NOT IN BPR:",
    len(
        top5_ids - bpr_item_ids
    )
)

print("\nPRODUCTS IN BPR:")

print(
    list(matched_top5)
)

TOP 5 PRODUCTS: 5
TOP 5 IN BPR: 4
TOP 5 NOT IN BPR: 1

PRODUCTS IN BPR:
['amazon_B011BRUOMO', 'amazon_B074B28H3S', 'amazon_B0BZ4MFNY1', 'amazon_B073JWXGNT']


In [33]:
import numpy as np

user_to_idx = cf_mapping["user_to_idx"]
item_to_idx = cf_mapping["item_to_idx"]

def get_bpr_scores(user_id, product_ids):
    """
    Trả về điểm BPR cho các product_id.

    Sản phẩm không có trong BPR -> NaN.
    User không có trong BPR -> toàn bộ NaN.
    """

    scores = np.full(
        len(product_ids),
        np.nan,
        dtype=np.float32
    )

    if user_id not in user_to_idx:
        return scores

    user_idx = int(
        user_to_idx[user_id]
    )

    user_vector = (
        bpr_model.user_factors[user_idx]
    )

    for position, product_id in enumerate(
        product_ids
    ):
        item_idx = item_to_idx.get(
            product_id
        )

        if item_idx is None:
            continue

        item_vector = (
            bpr_model.item_factors[
                int(item_idx)
            ]
        )

        scores[position] = float(
            np.dot(
                user_vector,
                item_vector
            )
        )

    return scores

In [34]:
def hybrid_rank_products(
    candidates,
    user_id=None,
    top_k=5
):
    # Xếp hạng semantic + rating + reviews
    result = rank_products(
        candidates,
        top_k=len(candidates)
    ).copy()

    if result.empty:
        return result

    result["bpr_score"] = np.nan

    result["hybrid_score"] = (
        result["final_score"]
    )

    result["ranking_method"] = (
        "semantic"
    )

    if (
        user_id is None
        or user_id not in user_to_idx
    ):
        return (
            result
            .head(top_k)
            .reset_index(drop=True)
        )

    bpr_scores = get_bpr_scores(
        user_id,
        result["product_id"].tolist()
    )

    result["bpr_score"] = bpr_scores

    valid = result[
        "bpr_score"
    ].notna()

    if valid.sum() >= 2:

        valid_scores = result.loc[
            valid,
            "bpr_score"
        ]

        minimum = valid_scores.min()
        maximum = valid_scores.max()

        if maximum > minimum:

            normalized = (
                valid_scores - minimum
            ) / (
                maximum - minimum
            )

        else:
            normalized = pd.Series(
                0.5,
                index=valid_scores.index
            )

        result.loc[
            valid,
            "hybrid_score"
        ] = (
            0.70
            *
            result.loc[
                valid,
                "final_score"
            ]
            +
            0.30
            *
            normalized
        )

        result.loc[
            valid,
            "ranking_method"
        ] = "semantic_bpr"

    result = result.sort_values(
        by=[
            "hybrid_score",
            "semantic_score",
            "product_id"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )

    return result.head(
        top_k
    ).reset_index(
        drop=True
    )

In [35]:
test_user_id = next(
    iter(user_to_idx)
)

print("TEST USER:", test_user_id)

hybrid_top5 = hybrid_rank_products(
    budget_candidates,
    user_id=test_user_id,
    top_k=5
)

display(
    hybrid_top5[
        [
            "product_id",
            "product_name",
            "semantic_score",
            "final_score",
            "bpr_score",
            "hybrid_score",
            "ranking_method"
        ]
    ]
)

print(
    "\nBPR PRODUCTS IN TOP 5:",
    hybrid_top5[
        "bpr_score"
    ].notna().sum()
)

TEST USER: amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA


,product_id,product_name,semantic_score,final_score,bpr_score,hybrid_score,ranking_method
0,amazon_B011BRUOMO,SanDisk Ultra 32GB microSDHC UHS-I Card with A...,0.779868,0.816060,0.336022,0.871242,semantic_bpr
1,amazon_B004RYB2EI,Sandisk 32GB Class 4 MicroSDHC MicroSD C4 TF F...,0.713282,0.737727,0.245535,0.786252,semantic_bpr
2,amazon_B074B28H3S,SanDisk Ultra SDSQUNS-032G-GN3MN 32GB 80MB/s U...,0.725661,0.772164,0.124373,0.769977,semantic_bpr
3,amazon_B01BL7CCBA,Verbatim 32GB Premium microSDHC Memory Card wi...,0.718353,0.761045,0.142650,0.768285,semantic_bpr
4,amazon_B016A5NH6C,ADATA Premier 32GB microSDHC/SDXC UHS-I Class ...,0.733113,0.765133,NaN,0.765133,semantic



BPR PRODUCTS IN TOP 5: 4


In [36]:
import numpy as np
import pandas as pd

def hybrid_rank_products(
    candidates,
    user_id=None,
    top_k=5
):
    result = rank_products(
        candidates,
        top_k=len(candidates)
    ).copy()

    if result.empty:
        return result

    result["bpr_score"] = np.nan
    result["bpr_adjustment"] = 0.0
    result["hybrid_score"] = result["final_score"]
    result["ranking_method"] = "semantic"

    # User mới hoặc không có trong BPR
    if (
        user_id is None
        or user_id not in user_to_idx
    ):
        return result.head(top_k).reset_index(
            drop=True
        )

    result["bpr_score"] = get_bpr_scores(
        user_id,
        result["product_id"].tolist()
    )

    valid = result["bpr_score"].notna()

    if valid.sum() >= 2:

        # Xếp hạng BPR trong các sản phẩm có điểm.
        # Chỉ dùng làm tín hiệu điều chỉnh nhỏ.
        bpr_percentile = (
            result.loc[valid, "bpr_score"]
            .rank(
                method="average",
                pct=True
            )
        )

        centered = (
            bpr_percentile
            -
            bpr_percentile.mean()
        )

        # Mức điều chỉnh tối đa không vượt 0.04
        adjustment = (
            0.04 * centered
        )

        result.loc[
            valid,
            "bpr_adjustment"
        ] = adjustment

        result.loc[
            valid,
            "hybrid_score"
        ] = (
            result.loc[valid, "final_score"]
            +
            adjustment
        )

        result.loc[
            valid,
            "ranking_method"
        ] = "semantic_bpr"

    result = result.sort_values(
        by=[
            "hybrid_score",
            "semantic_score",
            "product_id"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )

    return result.head(
        top_k
    ).reset_index(drop=True)

In [37]:
def recommend_products(
    user_query,
    user_id=None,
    top_k=5,
    candidate_k=200
):

    capacity = extract_capacity_gb(
        user_query
    )

    budget = extract_budget(
        user_query
    )

    if budget["needs_currency_clarification"]:

        return {
            "query": user_query,
            "user_id": user_id,
            "status": "needs_clarification",
            "message": (
                "Bạn muốn ngân sách "
                f"{budget['max_price']:,.0f} "
                "theo đơn vị tiền tệ nào "
                "(VND, USD hoặc TWD)?"
            ),
            "products": pd.DataFrame()
        }

    # FAISS
    candidates = retrieve_candidates(
        user_query,
        top_k=candidate_k
    )

    # Lọc dung lượng
    candidates = filter_by_capacity(
        candidates,
        capacity
    )

    # Lọc ngân sách và tiền tệ
    candidates = filter_by_budget(
        candidates,
        max_price=budget["max_price"],
        currency=budget["currency"]
    )

    if candidates.empty:

        return {
            "query": user_query,
            "user_id": user_id,
            "status": "no_match",
            "message": (
                "Chưa tìm thấy sản phẩm đáp ứng "
                "các điều kiện trong tập ứng viên "
                "đã truy xuất. Kết quả này không "
                "có nghĩa toàn bộ dataset không "
                "có sản phẩm phù hợp."
            ),
            "products": pd.DataFrame()
        }

    # Hybrid Ranking
    ranked = hybrid_rank_products(
        candidates,
        user_id=user_id,
        top_k=top_k
    )

    status = (
        "success"
        if len(ranked) == top_k
        else "partial"
    )

    return {
        "query": user_query,
        "user_id": user_id,
        "status": status,
        "message": (
            f"Tìm thấy {len(ranked)} "
            "sản phẩm đáp ứng các bộ lọc."
        ),
        "products": ranked
    }

In [38]:
test_query = (
    "Tôi cần thẻ nhớ microSD 32GB "
    "dưới 15 USD để lưu video"
)

# Người dùng mới
new_user_result = recommend_products(
    test_query,
    user_id=None,
    top_k=5
)

# Người dùng có trong BPR
existing_user_result = recommend_products(
    test_query,
    user_id=test_user_id,
    top_k=5
)

columns = [
    "product_id",
    "product_name",
    "final_score",
    "bpr_score",
    "bpr_adjustment",
    "hybrid_score",
    "ranking_method"
]

print("===== NEW USER =====")

display(
    new_user_result["products"][columns]
)

print("\n===== EXISTING USER =====")

display(
    existing_user_result["products"][columns]
)

assert new_user_result["status"] == "success"

assert existing_user_result["status"] == "success"

assert (
    new_user_result["products"]["ranking_method"]
    == "semantic"
).all()

print("\nPASS: Semantic và Hybrid đều hoạt động.")

===== NEW USER =====


,product_id,product_name,final_score,bpr_score,bpr_adjustment,hybrid_score,ranking_method
0,amazon_B073JWXGNT,SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card...,0.798381,NaN,0.0,0.798381,semantic
1,amazon_B011BRUOMO,SanDisk Ultra 32GB microSDHC UHS-I Card with A...,0.786037,NaN,0.0,0.786037,semantic
2,amazon_B0C2VMSBGX,"Hugolog 32GB Micro SD Card, MicroSDXC UHS-I Me...",0.772212,NaN,0.0,0.772212,semantic
3,amazon_B016A5NH6C,ADATA Premier 32GB microSDHC/SDXC UHS-I Class ...,0.767298,NaN,0.0,0.767298,semantic
4,amazon_B0BVVJM2X8,"LaView 32GB Micro SD Card 2 Pack, Micro SDXC U...",0.764532,NaN,0.0,0.764532,semantic



===== EXISTING USER =====


,product_id,product_name,final_score,bpr_score,bpr_adjustment,hybrid_score,ranking_method
0,amazon_B011BRUOMO,SanDisk Ultra 32GB microSDHC UHS-I Card with A...,0.786037,0.336022,0.012222,0.798259,semantic_bpr
1,amazon_B073JWXGNT,SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card...,0.798381,-0.564135,-0.018889,0.779492,semantic_bpr
2,amazon_B0C2VMSBGX,"Hugolog 32GB Micro SD Card, MicroSDXC UHS-I Me...",0.772212,NaN,0.000000,0.772212,semantic
3,amazon_B00E56J7FY,Kingston Digital 32 GB microSDHC Class 10 UHS-...,0.750147,0.551465,0.018889,0.769036,semantic_bpr
4,amazon_B016A5NH6C,ADATA Premier 32GB microSDHC/SDXC UHS-I Class ...,0.767298,NaN,0.000000,0.767298,semantic



PASS: Semantic và Hybrid đều hoạt động.


In [39]:
import json
import pandas as pd
from pathlib import Path

OUT_DIR = Path("/kaggle/working/step12")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Lưu kết quả người dùng mới
new_user_result["products"].to_parquet(
    OUT_DIR / "semantic_top5_test.parquet",
    index=False
)

# Lưu kết quả người dùng có trong BPR
existing_user_result["products"].to_parquet(
    OUT_DIR / "hybrid_top5_test.parquet",
    index=False
)

# Cập nhật cấu hình cuối cùng của Step 12
step12_config = {
    "step": "step12_hybrid_product_recommendation",
    "status": "completed",
    "product_count": int(len(products)),
    "faiss_vector_count": int(index.ntotal),
    "embedding_model": MODEL_NAME,
    "candidate_k": 200,
    "top_k": 5,

    "semantic_ranking_weights": {
        "semantic": 0.75,
        "rating": 0.20,
        "review_count": 0.05
    },

    "hybrid_ranking": {
        "enabled": True,
        "method": "bounded_bpr_adjustment",
        "bpr_adjustment_coefficient": 0.04,
        "fallback": "semantic",
        "requires_known_user": True
    },

    "bpr": {
        "users": int(
            bpr_model.user_factors.shape[0]
        ),
        "items": int(
            bpr_model.item_factors.shape[0]
        ),
        "latent_dimension": int(
            bpr_model.item_factors.shape[1]
        ),
        "product_coverage_percent": round(
            len(matched_ids) / len(products) * 100,
            2
        )
    },

    "constraints": {
        "capacity": True,
        "budget": True,
        "currency": True,
        "automatic_currency_conversion": False
    },

    "review_rag_integrated": False,
    "knowledge_editing_integrated": False
}

with open(
    OUT_DIR / "step12_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        step12_config,
        f,
        ensure_ascii=False,
        indent=2
    )

print("PASS: Đã lưu kết quả Hybrid Step 12.")

PASS: Đã lưu kết quả Hybrid Step 12.


In [40]:
import zipfile
from pathlib import Path

OUT_DIR = Path("/kaggle/working/step12")
ZIP_PATH = Path("/kaggle/working/step12_final.zip")

required_files = [
    "step12_config.json",
    "step12_test_results.json",
    "semantic_top5_test.parquet",
    "hybrid_top5_test.parquet",
]

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for filename in required_files:

        path = OUT_DIR / filename

        assert path.exists(), (
            f"Thiếu file: {filename}"
        )

        zf.write(
            path,
            arcname=filename
        )

print("===== STEP 12 OUTPUT =====")

with zipfile.ZipFile(ZIP_PATH) as zf:
    for name in zf.namelist():
        print("-", name)

print("\nSAVED:", ZIP_PATH)

print(
    "SIZE:",
    round(
        ZIP_PATH.stat().st_size / 1024**2,
        2
    ),
    "MB"
)

print("\nPASS: STEP 12 COMPLETED")

===== STEP 12 OUTPUT =====
- step12_config.json
- step12_test_results.json
- semantic_top5_test.parquet
- hybrid_top5_test.parquet

SAVED: /kaggle/working/step12_final.zip
SIZE: 0.03 MB

PASS: STEP 12 COMPLETED
